In [ ]:
import pandas as pd

master_df = pd.read_csv("D:/operational-signal-intelligence-environment/data/processed/master_operational_dataset.csv",
                        parse_dates=["Datetime"]
                        )

In [14]:
master_df["Renewable_share"]=(
    master_df["RENEWABLE"]
    / master_df["GENERATION"]
)*100

In [15]:
demand_threshold=master_df["ND"].quantile(0.95)

master_df["Demand_Spike"]=(
    master_df["ND"]>demand_threshold
)

In [16]:
master_df["Forecast_Failure"]=(
    master_df["Absolute_Error"]>2000
)

In [17]:
master_df["Low_Renewable_Window"]=(
    master_df["Renewable_share"]<20
)

In [18]:
master_df["High_Carbon_Period"]=(
    master_df["CARBON_INTENSITY"]>200
)

In [19]:
master_df["Supply_Stress"] = (
    master_df["ND"]
    >
    master_df["GENERATION"] * 0.95
)

In [20]:
master_df["Generation_Surplus"] = (
    master_df["GENERATION"]
    >
    master_df["ND"] * 1.20
)

In [21]:
renewable_threshold = (
    master_df["RENEWABLE"]
    .quantile(0.90)
)

master_df["Renewable_Surge"] = (
    master_df["RENEWABLE"]
    >
    renewable_threshold
)

In [22]:
low_demand_threshold = (
    master_df["ND"]
    .quantile(0.10)
)

master_df["Demand_Drop"] = (
    master_df["ND"]
    <
    low_demand_threshold
)

In [23]:
signal_summary = pd.DataFrame({
    "Signal":[
        "Demand Spike",
        "Forecast Failure",
        "Low Renewable Window",
        "High Carbon Period",
        "Supply Stress",
        "Generation Surplus",
        "Renewable Surge",
        "Demand Drop"
    ],
    "Count":[
        master_df["Demand_Spike"].sum(),
        master_df["Forecast_Failure"].sum(),
        master_df["Low_Renewable_Window"].sum(),
        master_df["High_Carbon_Period"].sum(),
        master_df["Supply_Stress"].sum(),
        master_df["Generation_Surplus"].sum(),
        master_df["Renewable_Surge"].sum(),
        master_df["Demand_Drop"].sum()
    ]
})

signal_summary

,Signal,Count
0,Demand Spike,876
1,Forecast Failure,718
2,Low Renewable Window,2759
3,High Carbon Period,2270
4,Supply Stress,601
5,Generation Surplus,10181
6,Renewable Surge,1752
7,Demand Drop,1752


In [25]:
signal_summary.to_csv(
    "D:/operational-signal-intelligence-environment/outputs/signal_summary.csv",
    index=False
)

In [26]:
master_df.to_csv(
    "D:/operational-signal-intelligence-environment/data/processed/master_with_signals.csv",
    index=False
)

In [27]:
signal_records = []

Demand Spike

In [28]:
for _, row in master_df[master_df["Demand_Spike"]].iterrows():

    signal_records.append({
        "timestamp": row["Datetime"],
        "signal_name": "Demand Spike",
        "severity": "High",
        "reason": "Demand exceeded spike threshold",
        "supporting_metric": row["ND"],
        "confidence": "High"
    })

Forecast Failure

In [29]:
for _, row in master_df[master_df["Forecast_Failure"]].iterrows():

    signal_records.append({
        "timestamp": row["Datetime"],
        "signal_name": "Forecast Failure",
        "severity": "Medium",
        "reason": "Forecast error exceeded threshold",
        "supporting_metric": row["Absolute_Error"],
        "confidence": "High"
    })

Low Renewable Window

In [30]:
for _, row in master_df[
    master_df["Low_Renewable_Window"]
].iterrows():

    signal_records.append({
        "timestamp": row["Datetime"],
        "signal_name": "Low Renewable Window",
        "severity": "Medium",
        "reason": "Renewable generation below threshold",
        "supporting_metric": row["RENEWABLE"],
        "confidence": "High"
    })

High Carbon Period

In [31]:
for _, row in master_df[
    master_df["High_Carbon_Period"]
].iterrows():

    signal_records.append({
        "timestamp": row["Datetime"],
        "signal_name": "High Carbon Period",
        "severity": "High",
        "reason": "Carbon intensity exceeded threshold",
        "supporting_metric": row["CARBON_INTENSITY"],
        "confidence": "High"
    })

Supply Stress

In [32]:
for _, row in master_df[
    master_df["Supply_Stress"]
].iterrows():

    signal_records.append({
        "timestamp": row["Datetime"],
        "signal_name": "Supply Stress",
        "severity": "High",
        "reason": "Demand approached generation capacity",
        "supporting_metric": row["ND"],
        "confidence": "Medium"
    })

Generation Surplus

In [33]:
for _, row in master_df[
    master_df["Generation_Surplus"]
].iterrows():

    signal_records.append({
        "timestamp": row["Datetime"],
        "signal_name": "Generation Surplus",
        "severity": "Low",
        "reason": "Generation significantly exceeded demand",
        "supporting_metric": row["GENERATION"],
        "confidence": "Medium"
    })

Renewable Surge

In [34]:
for _, row in master_df[
    master_df["Renewable_Surge"]
].iterrows():

    signal_records.append({
        "timestamp": row["Datetime"],
        "signal_name": "Renewable Surge",
        "severity": "Low",
        "reason": "Renewable generation in top 10%",
        "supporting_metric": row["RENEWABLE"],
        "confidence": "High"
    })

Demand Drop

In [35]:
for _, row in master_df[
    master_df["Demand_Drop"]
].iterrows():

    signal_records.append({
        "timestamp": row["Datetime"],
        "signal_name": "Demand Drop",
        "severity": "Low",
        "reason": "Demand in bottom 10%",
        "supporting_metric": row["ND"],
        "confidence": "High"
    })

In [36]:
signal_intelligence = pd.DataFrame(signal_records)

signal_intelligence.head()

,timestamp,signal_name,severity,reason,supporting_metric,confidence
0,2025-01-02 15:30:00,Demand Spike,High,Demand exceeded spike threshold,38264.0,High
1,2025-01-02 16:00:00,Demand Spike,High,Demand exceeded spike threshold,39309.0,High
2,2025-01-02 16:30:00,Demand Spike,High,Demand exceeded spike threshold,40412.0,High
3,2025-01-02 17:00:00,Demand Spike,High,Demand exceeded spike threshold,41103.0,High
4,2025-01-02 17:30:00,Demand Spike,High,Demand exceeded spike threshold,41589.0,High


In [37]:
signal_intelligence.to_csv(
    "D:/operational-signal-intelligence-environment/outputs/signal_intelligence_table.csv",
    index=False
)

In [38]:
signal_intelligence.shape

(20909, 6)